In [ ]:
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import time

# Set random seeds for reproducibility
def set_random_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# AutoEncoder class
class AutoEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Defining the Encoder
        self.encoder = nn.Sequential(
            # Input_dim -> 16 -> 8 -> 4 -> 2
            nn.Linear(input_dim, 16),
            nn.ELU(),
            nn.Linear(16, 8),
            nn.ELU(),
            nn.Linear(8, 4),
            nn.ELU(),
        )
        
        # Defining the Decoder
        self.decoder = nn.Sequential(
            nn.Linear(4, 8),
            nn.ELU(),
            nn.Linear(8, 16),
            nn.ELU(),
            nn.Linear(16, input_dim),
            nn.ELU(),
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    
    def encode(self, x):
        return self.encoder(x)

# Training function with early stopping
def train_autoencoder(model, train_loader, val_loader, epochs=100, lr=0.001, device='cpu'):
    # Setup
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Training trackers
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience = 15
    patience_counter = 0
    
    print(f"Training autoencoder for {epochs} epochs...")
    print("-" * 50)
    
    for epoch in range(epochs):
        start_time = time.time()
        
        # Training
        model.train()
        train_loss = 0.0
        for batch_data in train_loader:
            if isinstance(batch_data, (list, tuple)):
                batch_data = batch_data[0]  # Handle TensorDataset wrapper
            batch_data = batch_data.to(device)
            
            optimizer.zero_grad()
            reconstructed = model(batch_data)
            loss = criterion(reconstructed, batch_data)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_data in val_loader:
                if isinstance(batch_data, (list, tuple)):
                    batch_data = batch_data[0]  # Handle TensorDataset wrapper
                batch_data = batch_data.to(device)
                reconstructed = model(batch_data)
                loss = criterion(reconstructed, batch_data)
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model = model.state_dict().copy()
        else:
            patience_counter += 1
        
        # Print progress
        epoch_time = time.time() - start_time
        if (epoch + 1) % 20 == 0 or epoch < 5:
            print(f"Epoch {epoch+1:3d}/{epochs} | Train: {avg_train_loss:.6f} | "
                  f"Val: {avg_val_loss:.6f} | Time: {epoch_time:.1f}s | "
                  f"Patience: {patience_counter}/{patience}")
        
        # Stop if patience exceeded
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            print(f"Best validation loss: {best_val_loss:.6f}")
            model.load_state_dict(best_model)
            break
    
    print("Training completed!")
    return model, train_losses, val_losses

# LOHO CV with autoencoder training function
def leave_one_healthy_out_cv_autoencoder(patients, healthy_patient_ids, device='cpu', 
                                       epochs=100, batch_size=32, learning_rate=0.001):
    """
    LOHO CV specifically adapted for AutoEncoder models using your training function
    """
    cv_results = []
    
    for fold_idx, held_out_patient in enumerate(healthy_patient_ids):
        print(f"\n{'='*60}")
        print(f"FOLD {fold_idx + 1}/{len(healthy_patient_ids)}: Holding out patient {held_out_patient}")
        print(f"{'='*60}")
        
        # Prepare data
        training_patients = {pid: paths for pid, paths in patients.items() 
                           if pid != held_out_patient}
        validation_patient = {held_out_patient: patients[held_out_patient]}
        
        # Load and prepare training data
        training_dfs = []
        for patient_id, file_paths in training_patients.items():
            patient_data = pd.concat([
                pd.read_csv(file).drop(columns=['Time'], errors='ignore') 
                for file in file_paths
            ])
            training_dfs.append(patient_data)
        
        train_data = pd.concat(training_dfs, ignore_index=True)
        
        # Load and prepare validation data
        validation_dfs = []
        for patient_id, file_paths in validation_patient.items():
            patient_data = pd.concat([
                pd.read_csv(file).drop(columns=['Time'], errors='ignore') 
                for file in file_paths
            ])
            validation_dfs.append(patient_data)
        
        val_data = pd.concat(validation_dfs, ignore_index=True)
        
        # Remove target column if present (AutoEncoder is unsupervised)
        if 'target' in train_data.columns or 'label' in train_data.columns:
            target_col = 'target' if 'target' in train_data.columns else 'label'
            train_features = train_data.drop(columns=[target_col])
            val_features = val_data.drop(columns=[target_col])
        else:
            # Assume last column is target
            train_features = train_data.iloc[:, :-1]
            val_features = val_data.iloc[:, :-1]
        
        print(f"Training samples: {len(train_features)}, Validation samples: {len(val_features)}")
        print(f"Feature dimensions: {train_features.shape[1]}")
        
        # Standardize features
        scaler = StandardScaler()
        train_features_scaled = scaler.fit_transform(train_features)
        val_features_scaled = scaler.transform(val_features)
        
        # Convert to PyTorch tensors
        X_train = torch.FloatTensor(train_features_scaled)
        X_val = torch.FloatTensor(val_features_scaled)
        
        # Create DataLoaders
        train_dataset = TensorDataset(X_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        val_dataset = TensorDataset(X_val)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Initialize AutoEncoder
        input_dim = X_train.shape[1]
        autoencoder = AutoEncoder(input_dim)
        print(f"Model created with {sum(p.numel() for p in autoencoder.parameters())} parameters")
        
        # Train using your training function
        trained_model, train_history, val_history = train_autoencoder(
            model=autoencoder,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=epochs,
            lr=learning_rate,
            device=device
        )
        
        # Evaluate model - calculate reconstruction errors
        trained_model.eval()
        with torch.no_grad():
            # Calculate reconstruction errors
            train_reconstructions = []
            train_originals = []
            for batch_X, in train_loader:
                batch_X = batch_X.to(device)
                reconstructed = trained_model(batch_X)
                train_reconstructions.append(reconstructed.cpu().numpy())
                train_originals.append(batch_X.cpu().numpy())
            
            train_reconstructions = np.vstack(train_reconstructions)
            train_originals = np.vstack(train_originals)
            
            val_reconstructions = []
            val_originals = []
            for batch_X, in val_loader:
                batch_X = batch_X.to(device)
                reconstructed = trained_model(batch_X)
                val_reconstructions.append(reconstructed.cpu().numpy())
                val_originals.append(batch_X.cpu().numpy())
            
            val_reconstructions = np.vstack(val_reconstructions)
            val_originals = np.vstack(val_originals)
        
        # Calculate reconstruction errors (MSE)
        train_mse = mean_squared_error(train_originals, train_reconstructions)
        val_mse = mean_squared_error(val_originals, val_reconstructions)
        
        # Calculate per-sample reconstruction errors for anomaly detection
        train_sample_errors = np.mean((train_originals - train_reconstructions) ** 2, axis=1)
        val_sample_errors = np.mean((val_originals - val_reconstructions) ** 2, axis=1)
        
        # Calculate anomaly detection threshold
        threshold = np.mean(train_sample_errors) + 2 * np.std(train_sample_errors)
        val_anomalies = np.sum(val_sample_errors > threshold)
        
        fold_result = {
            'fold': fold_idx + 1,
            'held_out_patient': held_out_patient,
            'train_mse': train_mse,
            'val_mse': val_mse,
            'train_sample_errors': train_sample_errors,
            'val_sample_errors': val_sample_errors,
            'train_error_mean': np.mean(train_sample_errors),
            'train_error_std': np.std(train_sample_errors),
            'val_error_mean': np.mean(val_sample_errors),
            'val_error_std': np.std(val_sample_errors),
            'anomaly_threshold': threshold,
            'val_anomalies': val_anomalies,
            'model': trained_model,
            'scaler': scaler,
            'train_losses': train_history,
            'val_losses': val_history,
            'final_train_loss': train_history[-1],
            'final_val_loss': val_history[-1],
            'epochs_trained': len(train_history)
        }
        
        cv_results.append(fold_result)
        
        # Print fold summary
        print(f"\nFOLD {fold_idx + 1} SUMMARY:")
        print(f"Final Train Loss: {fold_result['final_train_loss']:.6f}")
        print(f"Final Val Loss: {fold_result['final_val_loss']:.6f}")
        print(f"Train MSE: {train_mse:.6f}, Val MSE: {val_mse:.6f}")
        print(f"Train Error: {fold_result['train_error_mean']:.6f} ± {fold_result['train_error_std']:.6f}")
        print(f"Val Error: {fold_result['val_error_mean']:.6f} ± {fold_result['val_error_std']:.6f}")
        print(f"Anomaly Threshold: {threshold:.6f}")
        print(f"Val samples above threshold: {val_anomalies}/{len(val_sample_errors)} ({100*val_anomalies/len(val_sample_errors):.1f}%)")
        print(f"Epochs trained: {fold_result['epochs_trained']}")
    
    return cv_results

def summarize_autoencoder_cv_results(cv_results):
    """
    Summarize AutoEncoder cross-validation results
    """
    train_mses = [result['train_mse'] for result in cv_results]
    val_mses = [result['val_mse'] for result in cv_results]
    
    train_error_means = [result['train_error_mean'] for result in cv_results]
    val_error_means = [result['val_error_mean'] for result in cv_results]
    
    train_losses = [result['final_train_loss'] for result in cv_results]
    val_losses = [result['final_val_loss'] for result in cv_results]
    
    summary = {
        'mean_train_mse': np.mean(train_mses),
        'std_train_mse': np.std(train_mses),
        'mean_val_mse': np.mean(val_mses),
        'std_val_mse': np.std(val_mses),
        'mean_train_error': np.mean(train_error_means),
        'std_train_error': np.std(train_error_means),
        'mean_val_error': np.mean(val_error_means),
        'std_val_error': np.std(val_error_means),
        'mean_final_train_loss': np.mean(train_losses),
        'std_final_train_loss': np.std(train_losses),
        'mean_final_val_loss': np.mean(val_losses),
        'std_final_val_loss': np.std(val_losses)
    }
    
    print("\n" + "="*60)
    print("AUTOENCODER LEAVE ONE HEALTHY OUT CV SUMMARY")
    print("="*60)
    print(f"Mean Training MSE: {summary['mean_train_mse']:.6f} ± {summary['std_train_mse']:.6f}")
    print(f"Mean Validation MSE: {summary['mean_val_mse']:.6f} ± {summary['std_val_mse']:.6f}")
    print(f"Mean Training Error: {summary['mean_train_error']:.6f} ± {summary['std_train_error']:.6f}")
    print(f"Mean Validation Error: {summary['mean_val_error']:.6f} ± {summary['std_val_error']:.6f}")
    print(f"Mean Final Train Loss: {summary['mean_final_train_loss']:.6f} ± {summary['std_final_train_loss']:.6f}")
    print(f"Mean Final Val Loss: {summary['mean_final_val_loss']:.6f} ± {summary['std_final_val_loss']:.6f}")
    
    # Higher validation error might indicate better anomaly detection capability
    if summary['mean_val_error'] > summary['mean_train_error']:
        print(f"\nGood! Validation error is higher than training error.")
        print("This suggests the model can distinguish between training and new healthy data.")
    
    return summary

# Test with unhealthy patients
def test_on_unhealthy_patients(cv_results, patients, unhealthy_patient_ids, device='cpu'):
    """
    Test the trained models on unhealthy patients to evaluate anomaly detection
    """
    print("\n" + "="*60)
    print("TESTING ON UNHEALTHY PATIENTS")
    print("="*60)
    
    unhealthy_results = []
    
    for unhealthy_id in unhealthy_patient_ids:
        print(f"\nTesting on unhealthy patient {unhealthy_id}:")
        
        # Load unhealthy patient data
        unhealthy_dfs = []
        for file_path in patients[unhealthy_id]:
            patient_data = pd.read_csv(file_path).drop(columns=['Time'], errors='ignore')
            unhealthy_dfs.append(patient_data)
        
        unhealthy_data = pd.concat(unhealthy_dfs, ignore_index=True)
        
        # Remove target column if present
        if 'target' in unhealthy_data.columns or 'label' in unhealthy_data.columns:
            target_col = 'target' if 'target' in unhealthy_data.columns else 'label'
            unhealthy_features = unhealthy_data.drop(columns=[target_col])
        else:
            unhealthy_features = unhealthy_data.iloc[:, :-1]
        
        fold_results = []
        
        # Test with each fold's model
        for fold_result in cv_results:
            model = fold_result['model']
            scaler = fold_result['scaler']
            
            # Preprocess data
            unhealthy_scaled = scaler.transform(unhealthy_features)
            X_unhealthy = torch.FloatTensor(unhealthy_scaled)
            
            # Calculate reconstruction errors
            model.eval()
            with torch.no_grad():
                X_unhealthy = X_unhealthy.to(device)
                reconstructed = model(X_unhealthy)
                reconstruction_errors = torch.mean((X_unhealthy - reconstructed) ** 2, dim=1).cpu().numpy()
            
            # Check how many samples are above the threshold
            threshold = fold_result['anomaly_threshold']
            anomalies = np.sum(reconstruction_errors > threshold)
            anomaly_rate = anomalies / len(reconstruction_errors)
            
            fold_results.append({
                'fold': fold_result['fold'],
                'held_out_patient': fold_result['held_out_patient'],
                'reconstruction_errors': reconstruction_errors,
                'threshold': threshold,
                'anomalies': anomalies,
                'anomaly_rate': anomaly_rate,
                'mean_error': np.mean(reconstruction_errors),
                'std_error': np.std(reconstruction_errors)
            })
        
        unhealthy_results.append({
            'unhealthy_patient_id': unhealthy_id,
            'fold_results': fold_results
        })
        
        # Print summary for this unhealthy patient
        mean_anomaly_rate = np.mean([fr['anomaly_rate'] for fr in fold_results])
        mean_error = np.mean([fr['mean_error'] for fr in fold_results])
        
        print(f"  Mean anomaly detection rate: {mean_anomaly_rate:.3f} ({mean_anomaly_rate*100:.1f}%)")
        print(f"  Mean reconstruction error: {mean_error:.6f}")
        print(f"  Sample size: {len(unhealthy_features)}")
    
    return unhealthy_results

# Your specific dataset configuration
patients = {
    1: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data1a.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data1b.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data1c.csv'],
    2: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data2a.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data2b.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data2c.csv'],
    3: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data3a.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data3b.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data3c.csv'],
    4: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data4a.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data4b.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data4c.csv'],
    5: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data5a.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data5b.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data5c.csv'],
    6: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data6a.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data6b.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data6c.csv'],
    7: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data7.csv'],
    8: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data8a.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data8b.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data8c.csv'],
    9: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data9a.csv',
        '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data9b.csv'],
    10: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data10a.csv',
         '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data10b.csv',
         '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data10c.csv'],
    11: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data11.csv'],
    12: ['/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data12a.csv',
         '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data12b.csv',
         '/Users/subhadyutirath/Desktop/FlowCytometryPytorch/data/organized_fcs_data12c.csv']
}

# Define healthy patients (1-6) and unhealthy patients (7-12)
healthy_patient_ids = [1, 2, 3, 4, 5, 6]
unhealthy_patient_ids = [7, 8, 9, 10, 11, 12]

if __name__ == "__main__":
    # Set random seeds for reproducibility
    set_random_seeds(42)
    
    # Determine device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    # Run LOHO cross-validation for AutoEncoder
    print("Starting Leave One Healthy Out Cross-Validation...")
    cv_results = leave_one_healthy_out_cv_autoencoder(
        patients=patients,
        healthy_patient_ids=healthy_patient_ids,
        device=device,
        epochs=100,
        batch_size=32,
        learning_rate=0.001
    )
    
    # Summarize results across all folds
    print("\nSummarizing cross-validation results...")
    summary = summarize_autoencoder_cv_results(cv_results)
    
    # Test on unhealthy patients for anomaly detection evaluation
    print("\nTesting anomaly detection on unhealthy patients...")
    unhealthy_results = test_on_unhealthy_patients(
        cv_results=cv_results,
        patients=patients,
        unhealthy_patient_ids=unhealthy_patient_ids,
        device=device
    )
    
    # Print final summary
    print(f"\n{'='*60}")
    print("FINAL SUMMARY")
    print(f"{'='*60}")
    print(f"Completed {len(cv_results)} folds of LOHO cross-validation")
    print(f"Tested on {len(unhealthy_patient_ids)} unhealthy patients")
    
    # Calculate overall anomaly detection performance
    all_anomaly_rates = []
    for unhealthy_result in unhealthy_results:
        for fold_result in unhealthy_result['fold_results']:
            all_anomaly_rates.append(fold_result['anomaly_rate'])
    
    mean_anomaly_detection_rate = np.mean(all_anomaly_rates)
    std_anomaly_detection_rate = np.std(all_anomaly_rates)
    
    print(f"Overall anomaly detection rate: {mean_anomaly_detection_rate:.3f} ± {std_anomaly_detection_rate:.3f}")
    print(f"({mean_anomaly_detection_rate*100:.1f}% ± {std_anomaly_detection_rate*100:.1f}%)")
    
    # Access individual results
    print(f"\nIndividual fold results available in 'cv_results' variable")
    print(f"Unhealthy patient test results available in 'unhealthy_results' variable")
    print(f"Summary statistics available in 'summary' variable")

Using device: cpu
Starting Leave One Healthy Out Cross-Validation...

FOLD 1/6: Holding out patient 1
Training samples: 43627497, Validation samples: 4248575
Feature dimensions: 13
Model created with 801 parameters
Training autoencoder for 100 epochs...
--------------------------------------------------
